# §1 体验实验：0.1B 模型开口说话

这个笔记本在你的电脑（或 Colab 的免费 CPU）上加载一个**只有 1 亿参数**的小模型 MiniMind2，让它用中文回答同一个开头，并对比两档"温度"（temperature）下输出的差别。

- **模型**：MiniMind2（约 104M 参数，Llama 架构，已做指令微调）——本书 §7 的结业项目就是它的家族。
- **实验**：开头固定为「给我讲讲猫」，温度 0.2 与 1.5 各采样 3 次，每次最多生成 60 个 token。
- **安装说明**：Colab 已自带 PyTorch，只需运行下面第一格安装 `transformers` 和 `modelscope`。本地运行请先 `pip install torch`（CPU 版即可）。

本笔记本中的代码与输出均来自 2026-10-08 的真实运行记录。

In [ ]:
# 安装依赖（Colab 自带 torch，无需重装；本地请先 pip install torch）
%pip install -q "transformers>=4.56" modelscope

## 1. 下载并加载模型

权重只有约 208 MB，普通 CPU 几秒就能加载完。下面默认走国内可达的 **ModelScope**；如果你能访问 Hugging Face，注释里也给出了对应方案。

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_DIR = './minimind2'

# 方案一（默认，国内可达）：从 ModelScope 下载
from modelscope import snapshot_download
snapshot_download('gongjy/minimind2', local_dir=MODEL_DIR)

# 方案二（huggingface.co 可达时使用；本机实测网络不可达，未验证）
# from huggingface_hub import snapshot_download
# snapshot_download('jingyaogong/MiniMind2', local_dir=MODEL_DIR)

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.float32)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'参数量: {n_params/1e6:.1f}M | 加载耗时: {time.time()-t0:.1f}s')

## 2. 单次生成：先听它开口

MiniMind2 是指令微调模型，输入要走它的 chat template（模板会自动加上 `<|im_start|>` 等对话标记）。

In [ ]:
PROMPT = '给我讲讲猫'

messages = [{'role': 'user', 'content': PROMPT}]
prompt_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print('模板化后的输入:', repr(prompt_text))
input_ids = tokenizer(prompt_text, return_tensors='pt').input_ids

torch.manual_seed(42)
t0 = time.time()
with torch.no_grad():
    out = model.generate(input_ids, max_new_tokens=60, do_sample=True,
                         temperature=0.2, pad_token_id=tokenizer.eos_token_id)
print(f'\n生成耗时: {time.time()-t0:.1f}s')
print('输出:', tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True))

## 3. 两档温度对比：0.2 vs 1.5

同一个开头，温度 0.2（保守）与 1.5（奔放）各采样 3 次。温度越低，模型越倾向选概率最高的词，输出越稳定雷同；温度越高，小概率词也会被抽中，输出越发散、也更容易出错。下面的输出是真实运行留下的原文（含截断产生的乱码，未做任何修饰）。

In [ ]:
for temp in [0.2, 1.5]:
    for i in range(3):
        seed = 42 + i
        torch.manual_seed(seed)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(input_ids, max_new_tokens=60, do_sample=True,
                                 temperature=temp, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
        print(f'--- 温度={temp} 第{i+1}次 (seed={seed}, {time.time()-t0:.1f}s) ---')
        print(text)
        print()

## 4. 观察要点

- 温度 0.2 的三条输出几乎都以「猫，作为人类……」开头，是稳定的"百科腔"，但会出现「可爱性和可爱性」这类重复。
- 温度 1.5 的三条开头各不相同，出现了「哺乳动物中家伙」这样的语病和离题——高温带来的"野性"肉眼可见。
- 输出末尾偶尔出现 `�`：这是 60 个 token 的硬截断切断了一个多字节汉字，属于正常现象。
- 全部六条原始输出与运行环境见同目录《首次运行记录.md》。